<a href="https://colab.research.google.com/github/jcmachicao/MachineLearningAvanzado_UC_2025/blob/main/U2__estrategias_mejora_modelado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Notebook Interactivo: Estrategias para Mejorar Redes Neuronales

Este notebook permite experimentar con diferentes configuraciones: "escaso", "ruidoso", "desbalanceado"

In [ ]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------
# ⚙️ Configuración
# -----------------------------
config = "escaso"  # cambia a: "ruidoso" o "desbalanceado"

# -----------------------------
# 📊 Dataset
# -----------------------------
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

if config == "escaso":
    X, _, y, _ = train_test_split(X, y, train_size=0.2, random_state=42)

if config == "desbalanceado":
    # eliminar muchos ejemplos de una clase
    mask = y == 0
    X = X[mask | (torch.rand(len(y)) < 0.2).numpy()]
    y = y[mask | (torch.rand(len(y)) < 0.2).numpy()]

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

# -----------------------------
# 🧩 Modelo
# -----------------------------
class MLP(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            activation(),
            nn.Linear(32, 32),
            activation(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# 🔧 Configuración dinámica
# -----------------------------
if config == "escaso":
    activation = nn.LeakyReLU
    optimizer_fn = lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9)
    loss_fn = lambda logits, y: F.huber_loss(logits, F.one_hot(y, num_classes=2).float())

elif config == "ruidoso":
    activation = nn.ELU
    optimizer_fn = lambda params: torch.optim.RMSprop(params, lr=0.01)
    def loss_fn(logits, y):
        target = F.one_hot(y, num_classes=2).float()
        return torch.mean(torch.log(torch.cosh(logits - target)))

elif config == "desbalanceado":
    activation = nn.SiLU
    optimizer_fn = lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9, nesterov=True)
    def loss_fn(logits, y, alpha=0.25, gamma=2.0):
        ce = F.cross_entropy(logits, y, reduction="none")
        pt = torch.exp(-ce)
        return (alpha * (1 - pt) ** gamma * ce).mean()

# -----------------------------
# 🚀 Entrenamiento
# -----------------------------
model = MLP(activation)
optimizer = optimizer_fn(model.parameters())

train_losses = []
val_losses = []

for epoch in range(100):
    model.train()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val)

    train_losses.append(loss.item())
    val_losses.append(val_loss.item())

# -----------------------------
# 📈 Visualización
# -----------------------------
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.legend()
plt.title(f"Curvas de pérdida - config: {config}")
plt.show()

# -----------------------------
# 🧪 Experimento sugerido
# -----------------------------
# Cambia el valor de 'config' y observa:
# - Estabilidad del entrenamiento
# - Diferencia train vs validation
# - Velocidad de convergencia

print("Experimento completado para config:", config)
